### **Lesson 1: MCP-Model Context Protocol**

MCP (Model Context Protocol) is an open-source standard for connecting AI applications to external systems.
Using MCP, AI applications like Claude or ChatGPT can connect to data sources (e.g. local files, databases), tools (e.g. search engines, calculators) and workflows (e.g. specialized prompts)—enabling them to access key information and perform tasks.

- Build MCP server.
- How to connect with 3rd party server.

**MCP server:** /Users/hemantkp/Desktop/DataScience/GenAI/LANGCHAIN/mcp_server.py

In [3]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.7, #it handles the randomness of the output, higher values (e.g., 0.8) make the output more random, while lower values (e.g., 0.2) make it more focused and deterministic.
    max_tokens=2048, #it limits the maximum number of tokens in the generated response. This helps control the length of the output and can prevent excessively long responses. 
    timeout = 30, #it specifies the maximum time (in seconds) that the model will take to generate a response.If the model takes longer than this time, it will stop and return whatever it has generated so far. This is useful for preventing long waits in case of complex queries or issues with the model.
    max_retries=3, #it defines the maximum number of times to retry a request if it fails due to network issues or other transient errors. This helps improve the robustness of your application by allowing it to recover from temporary problems without crashing.
)

### **Local MCP server**

In [12]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# configure your MCP client to point at your running server
client = MultiServerMCPClient({
    "tool_server": {
        "transport": "stdio",
        "command": "python",
        "args": ["mcp_server.py"],
    }
})

# get tools
tools = await client.get_tools()

# get a prompt by server name and prompt name
prompt_messages = await client.get_prompt("tool_server", "prompt")
prompt_messages = prompt_messages[0].content  # extract the content from the message

# # show the prompt content
# print(prompt_messages)

Failed to parse JSONRPC message from server
Traceback (most recent call last):
  File "/Users/hemantkp/Desktop/DataScience/GenAI/LANGCHAIN/.venv/lib/python3.13/site-packages/mcp/client/stdio/__init__.py", line 155, in stdout_reader
    message = types.JSONRPCMessage.model_validate_json(line)
  File "/Users/hemantkp/Desktop/DataScience/GenAI/LANGCHAIN/.venv/lib/python3.13/site-packages/pydantic/main.py", line 766, in model_validate_json
    return cls.__pydantic_validator__.validate_json(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        json_data, strict=strict, extra=extra, context=context, by_alias=by_alias, by_name=by_name
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
pydantic_core._pydantic_core.ValidationError: 1 validation error for JSONRPCMessage
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='Starting MCP server (streamable HTTP)…', input_type=str]
    For further info

In [9]:
from langchain.agents import create_agent

# Create an agent with the LLM
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt_messages
    )

In [11]:
from pprint import pprint
from langchain.messages import AIMessage, HumanMessage

config = {
    "configurable": {"thread_id": "1"},
}


# prompt asking for a search + summary
response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="Use the web_search tool to find LangChain MCP adapter library docs and summarize key points."
            )
        ]
    }
)

pprint(response)

{'messages': [HumanMessage(content='Use the web_search tool to find LangChain MCP adapter library docs and summarize key points.', additional_kwargs={}, response_metadata={}, id='d5dcd3e6-82a7-4369-b789-41a4b6e1d559'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'h8hkzge4g', 'function': {'arguments': '{"query":"LangChain MCP adapter library docs"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 279, 'total_tokens': 299, 'completion_time': 0.037932274, 'completion_tokens_details': None, 'prompt_time': 0.050267279, 'prompt_tokens_details': None, 'queue_time': 0.061680321, 'total_time': 0.088199553}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019caabc-32cc-78e1-95a4-08f1d881ce3f-0', tool_calls=[{'name': 'web_search', 'args': {'qu

### **Online MCP server**

**TimeZone MCP server**

In [10]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
  {
    "time": {
      "transport": "stdio",
      "command": "uvx",
      "args": [
        "mcp-server-time",
        "--local-timezone=America/New_York"
      ]
    }
}
)

tools = await client.get_tools()

In [11]:
from pprint import pprint
from langchain.messages import AIMessage, HumanMessage
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that provides the current time using the time tool."
)

response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="What time is it in India right now?"
            )
        ]
    }
)               

pprint(response)

{'messages': [HumanMessage(content='What time is it in India right now?', additional_kwargs={}, response_metadata={}, id='a79781d6-899d-4278-904e-2e9222427605'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'abrz6ftyd', 'function': {'arguments': '{"timezone":"Asia/Kolkata"}', 'name': 'get_current_time'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 451, 'total_tokens': 468, 'completion_time': 0.044364418, 'completion_tokens_details': None, 'prompt_time': 0.033727949, 'prompt_tokens_details': None, 'queue_time': 0.160923851, 'total_time': 0.078092367}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cacc9-8fd4-7423-8392-b4b4fbe20dcf-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Kolkata'}, 'id': 'abrz6ftyd', 'type': 'tool_call'

**Flight Search**

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
  {
  "mcpServers": {
      "transport": "streamable_http",
      "url": "https://mcp.kiwi.com"
    }
  }
)
tools = await client.get_tools()

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=llm,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a helpful assistant that provides travel information using the travel_server tool."
)

In [13]:
from langchain.messages import HumanMessage
config = {
    "configurable": {"thread_id": "1"},
}
response = await agent.ainvoke(
    { "messages": [HumanMessage(content="What flights are available from New York to London next week?")]},
    config
)

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search-flight,{"flyFrom": "New York", "flyTo": "London", "departureDate": "10/03/2024", "returnDate": "17/03/2024"}</function>'}}

### **Lesson 2: Context and State**

### **Lesson 3: Multi-Agent Systems**

### **Lesson 4: Wedding Planner**